# TASK-002 — NVIDIA Korean Conformer-CTC Surface ASR Baseline

이 Notebook은 `busan-surface-v0@1.0.0`과 외부 NVIDIA Riva 실행 결과의 계약을 검증합니다. 공개 NGC 페이지에서 확인된 후보는 `RIVA Conformer ASR Korean`, `deployable_v1.0`, Korean `ko-KR`, Conformer-CTC입니다.

NGC 파일 브라우저는 로그인 뒤에만 열립니다. 따라서 정확한 배포 파일명, 파일 SHA-256과 호환 Riva 런타임은 `pending verification`입니다. 이 값들을 확인하기 전에는 `.nemo`, NeMo `restore_from`, 또는 영어 FastConformer 모델로 대체하지 않습니다.


In [ ]:
BENCHMARK_ID = "busan-surface-v0"
BENCHMARK_VERSION = "1.0.0"
EXPERIMENT_ID = "task-002-nvidia-korean-conformer-ctc-pretrained-v0"
MODEL_PROVIDER = "NVIDIA"
MODEL_NAME = "RIVA Conformer ASR Korean"
MODEL_FAMILY = "Conformer-CTC"
MODEL_VERSION = "deployable_v1.0"
CHECKPOINT_IDENTIFIER = "nvidia/tao/speechtotext_ko_kr_conformer:deployable_v1.0"
DECODER_TYPE = "CTC greedy"
FINE_TUNED = False

# NGC 인증 뒤 실제 다운로드·실행 환경에서 기록해야 하는 값입니다.
ARTIFACT_FILENAME = ""
ARTIFACT_SHA256 = ""
RIVA_RUNTIME_VERSION = ""

pending = [
    name
    for name, value in {
        "ARTIFACT_FILENAME": ARTIFACT_FILENAME,
        "ARTIFACT_SHA256": ARTIFACT_SHA256,
        "RIVA_RUNTIME_VERSION": RIVA_RUNTIME_VERSION,
    }.items()
    if not value
]
print("pending verification:", ", ".join(pending) or "none")


## 실행 경계

이 NGC 항목은 Riva Quick Start 사용 경로로 배포되며 일반 Python에서 `.nemo` 체크포인트를 직접 여는 모델로 확인되지 않았습니다. 실제 추론은 NVIDIA GPU가 있는 Linux x86_64/Riva 환경에서 수행하고, 이 Notebook에는 결과 `predictions.jsonl`과 Benchmark ZIP을 가져옵니다.

아래 검사는 추론을 대신하지 않습니다. 외부 실행 결과의 10개 ID·오디오 hash·모델 메타데이터가 고정 Benchmark와 일치하는지만 확인합니다.


In [ ]:
import json
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()  # Benchmark ZIP과 predictions.jsonl을 선택
bundle_names = [name for name in uploaded if name.endswith(".zip")]
prediction_names = [name for name in uploaded if name.endswith(".jsonl")]
if len(bundle_names) != 1 or len(prediction_names) != 1:
    raise RuntimeError("Upload exactly one benchmark ZIP and one predictions.jsonl")

with zipfile.ZipFile(bundle_names[0]) as bundle:
    manifest = json.loads(bundle.read("benchmark.json"))

if (
    manifest["benchmark_id"] != BENCHMARK_ID
    or manifest["benchmark_version"] != BENCHMARK_VERSION
    or manifest["frozen"] is not True
    or len(manifest["entries"]) != 10
):
    raise ValueError("unexpected benchmark identity or size")

predictions = [
    json.loads(line)
    for line in Path(prediction_names[0]).read_text(encoding="utf-8").splitlines()
    if line.strip()
]
expected = {
    (entry["utterance_id"], entry["derived_audio_sha256"])
    for entry in manifest["entries"]
}
actual = {(item["utterance_id"], item["audio_sha256"]) for item in predictions}
if len(predictions) != 10 or actual != expected:
    raise ValueError("predictions do not exactly match the frozen benchmark")

for item in predictions:
    if item["experiment_id"] != EXPERIMENT_ID:
        raise ValueError("unexpected experiment_id")
    model = item["result"]["model"]
    if model["name"] != MODEL_NAME or model["version"] != MODEL_VERSION:
        raise ValueError("unexpected model identity")
    if item["result"]["confidence_supported"] is False:
        assert item["result"]["confidence"] is None

print("Validated 10 predictions against busan-surface-v0@1.0.0")


Mac에서 Import와 평가:

```bash
python3 ~/.codex/skills/use-busan-project-venv/scripts/run.py busan-lab evaluate \
  --benchmark-id busan-surface-v0 \
  --benchmark-version 1.0.0 \
  --predictions /absolute/path/predictions.jsonl
```
